In [93]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

In [2]:
url = "https://volcano.si.edu/reports_weekly.cfm?vtab=feeds"

response = requests.get(url)
response

<Response [200]>

In [82]:
soup = BeautifulSoup(response.content)
table = soup.find('table')
table

<table role="presentation">
<tbody>
<tr>
<td class="varFigImage"><a href="/images/bulletin/241040/241040_BGVN_096.jpg"><img alt="Figure (see Caption)" src="/images/bulletin/241040/241040_BGVN_096.jpg" width="480"/></a></td>
<td class="varFigCaption"><span class="varCapNum">Figure 96.</span> Photo of a strong gas-and-steam plume rising above Whakaari/White Island on 28 May 2020. Courtesy of GeoNet.</td>
</tr>
</tbody>
</table>

In [5]:
# Extract column names
headers = [th.get_text(strip=True) for th in table.find_all('th')][1:]
headers

['Name', 'Country', 'Volcanic Region', 'Eruption Start Date', 'Report Status']

In [86]:
volcano_data = []
headers = [th.get_text(strip=True) for th in table.find_all('th')][1:]

for row in table.find_all('tr')[2:]:  # Skip header row
    cols = row.find_all(['td', 'th'])

    print(f"Row columns count: {len(cols)}")
    print(f"Row columns text: {[col.get_text(strip=True) for col in cols]}")
    
    if len(cols) < len(headers):
        print("Skipping row: not enough columns")
        continue

    try:
        volcano_link = row.find('a', href=re.compile(r'#vn_'))
        
        if not volcano_link:
            print("No volcano link found in this row")
            continue
        
        volcano_id = volcano_link['href'].split('#vn_')[1]
        volcano_name = volcano_link.get_text(strip=True)
        start_date = cols[3].get_text(strip=True)
        
        report_status = row.find("a", attrs={"data-tooltip": True})
        report_text = report_status.get_text(strip=True) if report_status else None

        row_data = {
              'volcano_id': volcano_id,
              'volcano_name': volcano_name,
              'start_date': start_date,
              'report_status': report_text
        }
        
        # # Add other column data
        # for i, col in enumerate(cols):
        #     if i < len(headers):
        #         row_data[headers[i]] = col.get_text(strip=True)
        
        volcano_data.append(row_data)
        print(f"Added data for: {volcano_name}")
    
    except Exception as e:
        print(f"Error processing row: {e}")

df = pd.DataFrame(volcano_data)
df

""


In [98]:
def scrap_volcanic_weekly_report():

    url = "https://volcano.si.edu/reports_weekly.cfm?vtab=feeds"
    response = requests.get(url)

    soup = BeautifulSoup(response.content)
    table = soup.find('table')

    volcano_data = []
    headers = [th.get_text(strip=True) for th in table.find_all('th')][1:]

    for row in table.find_all('tr')[2:]:  # Skip header row
        cols = row.find_all(['td', 'th'])

        if len(cols) < len(headers):
            print("Skipping row: not enough columns")
            continue

        try:
            volcano_link = row.find('a', href=re.compile(r'#vn_'))
            
            if not volcano_link:
                print("No volcano link found in this row")
                continue
            
            volcano_id = volcano_link['href'].split('#vn_')[1]
            volcano_name = volcano_link.get_text(strip=True)
            start_date = cols[3].get_text(strip=True)
            
            report_status = row.find("a", attrs={"data-tooltip": True})
            report_text = report_status.get_text(strip=True) if report_status else None

            row_data = {
                'volcano_id': volcano_id,
                'volcano_name': volcano_name,
                'start_date': start_date,
                'report_status': report_text
            }
            
            volcano_data.append(row_data)
        
        except Exception as e:
            print(f"Error processing row: {e}")

    volcanic_weekly_report = pd.DataFrame(volcano_data)

    return volcanic_weekly_report
    
volcanic_weekly_report = scrap_volcanic_weekly_report()
volcanic_weekly_report

,volcano_id,volcano_name,start_date,report_status
0,264180,Lewotobi,2023 Dec 23,New
1,264230,Lewotolok,2025 Jan 16,New
2,345040,Poas,2025 Jan 5,New
3,284141,Ahyi,2024 Aug 5,Continuing
4,282080,Aira,2017 Mar 25,Continuing
5,300250,Bezymianny,2024 Dec 24,Continuing
6,268010,Dukono,1933 Aug 13,Continuing
7,211060,Etna,2022 Nov 27,Continuing
8,311120,Great Sitkin,2021 May 25,Continuing
9,243080,Home Reef,2024 Dec 4,Continuing


In [9]:
volcano_link = row.find('a', href=re.compile(r'#vn_'))
volcano_link

<a href="#vn_241040">Whakaari/White Island</a>

In [10]:
row.find('a', href=re.compile(r'#vn_'))

<a href="#vn_241040">Whakaari/White Island</a>

In [11]:
volcano_number = volcano_link['href'].split('#vn_')[1]
volcano_number

'241040'

In [12]:
volcano_name = volcano_link.get_text(strip=True)
volcano_name

'Whakaari/White Island'

In [13]:
start_date = cols[3].get_text(strip=True)
start_date

'2024 May 24'

In [14]:
report_status = row.find("a", attrs={"data-tooltip": True})
report_status

<a data-tooltip="tt241040" style="color:green;">Continuing</a>

In [15]:
report_text = report_status.get_text(strip=True) if report_status else None
report_text

'Continuing'

In [16]:
volcano_data

[{'volcano_id': '264180',
  'volcano_name': 'Lewotobi',
  'start_date': '2023 Dec 23',
  'report_status': 'New'},
 {'volcano_id': '264230',
  'volcano_name': 'Lewotolok',
  'start_date': '2025 Jan 16',
  'report_status': 'New'},
 {'volcano_id': '345040',
  'volcano_name': 'Poas',
  'start_date': '2025 Jan 5',
  'report_status': 'New'},
 {'volcano_id': '284141',
  'volcano_name': 'Ahyi',
  'start_date': '2024 Aug 5',
  'report_status': 'Continuing'},
 {'volcano_id': '282080',
  'volcano_name': 'Aira',
  'start_date': '2017 Mar 25',
  'report_status': 'Continuing'},
 {'volcano_id': '300250',
  'volcano_name': 'Bezymianny',
  'start_date': '2024 Dec 24',
  'report_status': 'Continuing'},
 {'volcano_id': '268010',
  'volcano_name': 'Dukono',
  'start_date': '1933 Aug 13',
  'report_status': 'Continuing'},
 {'volcano_id': '211060',
  'volcano_name': 'Etna',
  'start_date': '2022 Nov 27',
  'report_status': 'Continuing'},
 {'volcano_id': '311120',
  'volcano_name': 'Great Sitkin',
  'start_d